# Greenhouse Slug Discovery

This notebook tests candidate Greenhouse board slugs against:

`https://boards-api.greenhouse.io/v1/boards/{slug}/jobs`

A slug is considered **working** when the API responds `200 OK`.

## Is this possible?

Yes. Greenhouse board URLs are public and can be probed by slug.

- `200` means the board exists and is reachable.
- `404` usually means no board for that slug.

> Note: this cannot guarantee a complete list of *all* companies using Greenhouse.
> It only validates the candidate slugs you provide.

In [ ]:
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable

import requests
import pandas as pd

In [ ]:
# --- Configuration ---
BASE_URL = "https://boards-api.greenhouse.io/v1/boards/{slug}/jobs"
MAX_WORKERS = 12
REQUEST_TIMEOUT = 12
SLEEP_JITTER_SEC = (0.0, 0.15)  # polite jitter between calls in each worker

# Seed list from the app's current curated companies.
seed_slugs = [
    "airbnb", "amplitude", "anthropic", "asana", "benchling", "brex", "canva", "cloudflare",
    "coinbase", "datadog", "databricks", "discord", "doordash", "dropbox", "duolingo",
    "figma", "fivetran", "gitlab", "gusto", "hashicorp", "hubspot", "intercom", "lattice",
    "launchdarkly", "linear", "loom", "lyft", "mercury", "miro", "mixpanel", "mongodb",
    "notion", "pagerduty", "palantir", "pinterest", "plaid", "ramp", "reddit", "retool",
    "rippling", "robinhood", "scale", "segment", "snowflake", "stripe", "twilio", "verkada",
    "zendesk", "zoom"
]

# Add your own guesses here (lowercase slug format):
extra_candidates = [
    # "openai", "instacart", "atlassian"
]

candidates = sorted(set(seed_slugs + extra_candidates))
print(f"Total candidate slugs: {len(candidates)}")

In [ ]:
def probe_slug(slug: str) -> dict:
    url = BASE_URL.format(slug=slug)
    t0 = time.perf_counter()
    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT)
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        # jitter to be polite when running high concurrency
        time.sleep(random.uniform(*SLEEP_JITTER_SEC))

        return {
            "slug": slug,
            "status_code": r.status_code,
            "ok": r.status_code == 200,
            "jobs_count": len(r.json().get("jobs", [])) if r.status_code == 200 else None,
            "url": url,
            "elapsed_ms": elapsed_ms,
        }
    except Exception as e:
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        return {
            "slug": slug,
            "status_code": None,
            "ok": False,
            "jobs_count": None,
            "url": url,
            "elapsed_ms": elapsed_ms,
            "error": str(e),
        }


def scan_slugs(slugs: Iterable[str]) -> pd.DataFrame:
    rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(probe_slug, s): s for s in slugs}
        for fut in as_completed(futures):
            rows.append(fut.result())
    df = pd.DataFrame(rows).sort_values(["ok", "status_code", "slug"], ascending=[False, True, True])
    return df.reset_index(drop=True)

In [ ]:
results = scan_slugs(candidates)
results.head(20)

In [ ]:
summary = (
    results.assign(status_group=results["status_code"].fillna(-1))
    .groupby("status_group", dropna=False)
    .size()
    .rename("count")
    .reset_index()
    .sort_values("status_group")
)
summary

In [ ]:
working = results[results["ok"]].copy()
print(f"Working slugs: {len(working)}")
working[["slug", "jobs_count", "status_code", "elapsed_ms"]].head(50)

In [ ]:
# Save results for later use.
output_csv = "greenhouse_slug_scan_results.csv"
results.sort_values(["ok", "slug"], ascending=[False, True]).to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")